# Data Preparation: Recipe → Sugar per Serving (NLP Assignment)

## Official Data Source
- Paper repository: https://github.com/minweiqing/You-Are-What-You-Eat-Exploring-Rich-Recipe-Information-for-Cross-Region-Food-Analysis
- Direct datasource (Google Drive): https://drive.google.com/open?id=1U5a-1R-b4Gl_La0rUAFWDgM8eIAF_pR3
- Dataset location in the archive: `Yummly-66K/food_min/Yummly28K.zip/metadata27638`
- Metadata files expected: `meta00001.json` ... `meta27638.json`

## NLP Data Preprocessing Pipeline
1. **Ingestion**: read each recipe JSON from `metadata27638`.
2. **Field extraction**: extract `ingredientLines`, `nutritionEstimates`, `numberOfServings`, `yield`, and source URL.
3. **Target engineering**: compute `sugar_total_g` from nutrition attribute `SUGAR`, then compute `sugar_per_serving_g = sugar_total_g / servings`.
4. **Serving normalization**: infer servings from `numberOfServings` first; fallback to parsing numeric value from `yield`.
5. **Text preparation**: build ingredient text by joining `ingredientLines` into one model input string.
6. **Data cleaning**: remove missing targets, keep non-negative values, and clip unrealistic outliers.
7. **Label scaling**: z-score normalize the target for model training (`sugar_per_serving_z`).

## NLP Techniques Used
- **Rule-based information extraction** from nested JSON fields.
- **Regex-based numeric parsing** to recover servings from free-text `yield`.
- **Text normalization / concatenation** for ingredient sequence modeling.
- **Supervised regression target transformation** with z-score normalization.

In [ ]:
pip install datasets huggingface_hub nltk spacy tokenizers transformers scikit-learn gdown

In [1]:
# Optional: install NLP dependencies if missing
import sys
import subprocess
import importlib.util

required_packages = {
    'datasets': 'datasets',
    'huggingface_hub': 'huggingface_hub',
    'nltk': 'nltk',
    'spacy': 'spacy',
    'tokenizers': 'tokenizers',
    'transformers': 'transformers',
    'sklearn': 'scikit-learn',
}

missing = [pip_name for module_name, pip_name in required_packages.items() if importlib.util.find_spec(module_name) is None]
if missing:
    print('Installing missing packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + missing)
else:
    print('All optional NLP packages already available.')

# Lightweight resource download for NLTK (safe to skip if unavailable)
try:
    import nltk
    for pkg in ['punkt', 'averaged_perceptron_tagger', 'wordnet', 'omw-1.4', 'stopwords']:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass
except Exception:
    pass

# Optional spaCy model download
try:
    import spacy
    try:
        spacy.load('en_core_web_sm')
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])
except Exception:
    print('spaCy model setup skipped.')

All optional NLP packages already available.


In [2]:
import os
import re
import json
from pathlib import Path

import tqdm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

C:\Users\SN798EZ\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Fast pre-check: skip download if metadata JSON already exists
from pathlib import Path

EXPECTED_JSON_COUNT = 27638
SEARCH_ROOTS = [Path('.'), Path('raw_yummly'), Path('downloads')]

existing_candidates = []
for root in SEARCH_ROOTS:
    if not root.exists():
        continue
    for folder in root.rglob('*'):
        if folder.is_dir():
            count = len(list(folder.glob('meta*.json')))
            if count > 0:
                existing_candidates.append((folder, count))

if existing_candidates:
    existing_candidates.sort(key=lambda x: x[1], reverse=True)
    best_dir, best_count = existing_candidates[0]
    DOWNLOADED_META_DIR = str(best_dir)
    SKIP_DATA_DOWNLOAD = best_count >= EXPECTED_JSON_COUNT
    print(f'Found existing metadata directory: {best_dir}')
    print(f'Found JSON count: {best_count}')
    print(f'Skip download: {SKIP_DATA_DOWNLOAD}')
else:
    DOWNLOADED_META_DIR = ''
    SKIP_DATA_DOWNLOAD = False
    print('No existing metadata directory found. Download will proceed.')

Found existing metadata directory: raw_yummly\metadata27638
Found JSON count: 27638
Skip download: True


In [7]:
# Download and prepare Yummly metadata JSON files from Google Drive
from pathlib import Path
import zipfile
import subprocess
import sys

FILE_ID = '1U5a-1R-b4Gl_La0rUAFWDgM8eIAF_pR3'
DOWNLOAD_DIR = Path('downloads')
EXTRACT_DIR = Path('raw_yummly')
OUT_FILE = DOWNLOAD_DIR / 'Yummly-66K.zip'

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

def _ensure_pkg(module_name, pip_name=None):
    pip_name = pip_name or module_name
    try:
        __import__(module_name)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

def _extract_zip(zip_path: Path, target_dir: Path):
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(target_dir)

# If the pre-check already found a complete metadata set, skip download/extract
if 'SKIP_DATA_DOWNLOAD' in globals() and SKIP_DATA_DOWNLOAD and 'DOWNLOADED_META_DIR' in globals() and DOWNLOADED_META_DIR:
    existing_count = len(list(Path(DOWNLOADED_META_DIR).glob('meta*.json')))
    print(f'Skipping download. Reusing existing metadata at: {DOWNLOADED_META_DIR}')
    print(f'Existing JSON file count: {existing_count}')
else:
    # 1) Download main archive
    _ensure_pkg('gdown')
    import gdown

    if not OUT_FILE.exists():
        print(f'Downloading to: {OUT_FILE}')
        gdown.download(id=FILE_ID, output=str(OUT_FILE), fuzzy=True, quiet=False)
    else:
        print(f'Using existing archive: {OUT_FILE}')

    # 2) Extract main archive
    main_extract = EXTRACT_DIR / 'Yummly-66K'
    if not main_extract.exists():
        print('Extracting main archive...')
        _extract_zip(OUT_FILE, EXTRACT_DIR)
    else:
        print(f'Using existing extracted directory: {main_extract}')

    # 3) Find and extract nested Yummly28K.zip
    inner_zips = list(EXTRACT_DIR.rglob('Yummly28K.zip'))
    if not inner_zips:
        raise FileNotFoundError('Yummly28K.zip not found inside downloaded archive.')

    inner_zip = inner_zips[0]
    inner_extract = EXTRACT_DIR / 'Yummly28K'
    if not inner_extract.exists():
        print(f'Extracting nested archive: {inner_zip}')
        _extract_zip(inner_zip, inner_extract)
    else:
        print(f'Using existing nested extraction: {inner_extract}')

    # 4) Locate metadata folder with meta*.json
    candidate_dirs = []
    for folder in EXTRACT_DIR.rglob('*'):
        if folder.is_dir():
            count = len(list(folder.glob('meta*.json')))
            if count > 0:
                candidate_dirs.append((folder, count))

    if not candidate_dirs:
        raise FileNotFoundError('No folder containing meta*.json was found after extraction.')

    candidate_dirs.sort(key=lambda x: x[1], reverse=True)
    best_meta_dir, file_count = candidate_dirs[0]
    DOWNLOADED_META_DIR = str(best_meta_dir)

    print('Detected metadata directory:', DOWNLOADED_META_DIR)
    print('Detected JSON file count:', file_count)
    print('Expected approximately 27,638 files.')

Skipping download. Reusing existing metadata at: raw_yummly\metadata27638
Existing JSON file count: 27638


In [8]:
# Set metadata directory (auto-use downloaded folder when available)
if 'DOWNLOADED_META_DIR' in globals() and DOWNLOADED_META_DIR and Path(DOWNLOADED_META_DIR).exists():
    META_DIR = Path(DOWNLOADED_META_DIR)
else:
    META_DIR = Path('meta')

# Optional fallback if folder is named metadata27638
if not META_DIR.exists() and Path('metadata27638').exists():
    META_DIR = Path('metadata27638')

data = sorted(str(p) for p in META_DIR.glob('meta*.json'))
print(f'Metadata directory: {META_DIR.resolve()}')
print(f'Found {len(data)} JSON files')
print('Sample:', data[:3])

Metadata directory: C:\Users\SN798EZ\Downloads\Projects\ingbetic-main\data\raw_yummly\metadata27638
Found 27638 JSON files
Sample: ['raw_yummly\\metadata27638\\meta00001.json', 'raw_yummly\\metadata27638\\meta00002.json', 'raw_yummly\\metadata27638\\meta00003.json']


In [9]:
grid = []
nutrition_desc = {}

def extract_servings(recipe_obj):
    number_of_servings = recipe_obj.get('numberOfServings')
    if isinstance(number_of_servings, (int, float)) and number_of_servings > 0:
        return float(number_of_servings)

    yield_text = recipe_obj.get('yield', '')
    if isinstance(yield_text, str):
        match = re.search(r'(\d+(?:\.\d+)?)', yield_text)
        if match:
            parsed = float(match.group(1))
            if parsed > 0:
                return parsed

    return np.nan

for d in tqdm.tqdm(data):
    row = {}

    with open(d, encoding='utf-8') as file:
        obj = json.load(file)

    source = obj.get('source', {})
    row['src'] = source.get('sourceRecipeUrl')
    row['recipe_name'] = obj.get('name')
    row['ingredients'] = ', '.join(obj.get('ingredientLines', []))

    sugar_total_g = np.nan
    nutrition = obj.get('nutritionEstimates', [])
    for n in nutrition:
        attribute = n.get('attribute')
        unit = n.get('unit', {})
        nutrition_desc[attribute] = {
            'unit': unit.get('plural') or unit.get('abbreviation'),
            'description': n.get('description'),
        }
        row[attribute] = n.get('value')

        if attribute == 'SUGAR':
            value = n.get('value')
            if isinstance(value, (int, float)):
                sugar_total_g = float(value)

    servings = extract_servings(obj)
    row['servings'] = servings
    row['sugar_total_g'] = sugar_total_g
    row['sugar_per_serving_g'] = (
        sugar_total_g / servings
        if pd.notna(sugar_total_g) and pd.notna(servings) and servings > 0
        else np.nan
    )

    grid.append(row)

100%|██████████| 27638/27638 [08:13<00:00, 55.98it/s]


In [10]:
df_raw = pd.DataFrame(data=grid)
df_raw.to_csv('data.csv', index=False)
df_raw[['recipe_name', 'servings', 'sugar_total_g', 'sugar_per_serving_g']].head()

,recipe_name,servings,sugar_total_g,sugar_per_serving_g
0,Mushroom Risotto,6.0,7.54,1.256667
1,Filipino BBQ Pork Skewers,4.0,16.65,4.162500
2,Mushroom and Roasted Garlic Risotto,1.0,52.20,52.200000
3,Gratin Dauphinois (Scalloped Potatoes with Che...,7.0,3.25,0.464286
4,Delicious Grilled Hamburgers,3.0,0.75,0.250000


In [11]:
df = pd.DataFrame(data=grid)
df = df[df['ingredients'].notna()]
df = df[df['sugar_per_serving_g'].notna()]
df = df[(df['sugar_per_serving_g'] >= 0) & (df['sugar_per_serving_g'] < 100)]
df = df[['src', 'recipe_name', 'ingredients', 'servings', 'sugar_total_g', 'sugar_per_serving_g']]
df.head()

,src,recipe_name,ingredients,servings,sugar_total_g,sugar_per_serving_g
0,http://www.skinnytaste.com/2009/10/risotto-is-...,Mushroom Risotto,"2 cups Baby Bella mushrooms, sliced, 2 cups ar...",6.0,7.54,1.256667
1,http://www.skinnytaste.com/2008/08/filipino-bb...,Filipino BBQ Pork Skewers,"2.5 lb pork country style ribs, all fat trimme...",4.0,16.65,4.162500
2,http://www.myrecipes.com/recipe/mushroom-roast...,Mushroom and Roasted Garlic Risotto,"2 whole garlic heads, 2 tablespoons plus 2 tea...",1.0,52.20,52.200000
3,http://www.myrecipes.com/recipe/gratin-dauphin...,Gratin Dauphinois (Scalloped Potatoes with Che...,"1 garlic clove, halved, Cooking spray, 6 peele...",7.0,3.25,0.464286
4,http://allrecipes.com/Recipe/delicious-grilled...,Delicious Grilled Hamburgers,"1 pound lean ground beef, 1 tablespoon Worcest...",3.0,0.75,0.250000


In [12]:
label = df['sugar_per_serving_g'].to_numpy(dtype=float)
mean = label.mean()
std = label.std()
mean, std

(3.0986566520133145, 7.514386460680026)

In [13]:
label = (label - mean) / std if std > 0 else np.zeros_like(label)
label.mean(), label.std()

(5.4670413782464984e-18, 1.0)

In [14]:
df['sugar_per_serving_z'] = label
df['sugar'] = label  # backward-compatible target name

In [15]:
df[['ingredients', 'servings', 'sugar_total_g', 'sugar_per_serving_g', 'sugar_per_serving_z']].head()

,ingredients,servings,sugar_total_g,sugar_per_serving_g,sugar_per_serving_z
0,"2 cups Baby Bella mushrooms, sliced, 2 cups ar...",6.0,7.54,1.256667,-0.245128
1,"2.5 lb pork country style ribs, all fat trimme...",4.0,16.65,4.162500,0.141574
2,"2 whole garlic heads, 2 tablespoons plus 2 tea...",1.0,52.20,52.200000,6.534312
3,"1 garlic clove, halved, Cooking spray, 6 peele...",7.0,3.25,0.464286,-0.350577
4,"1 pound lean ground beef, 1 tablespoon Worcest...",3.0,0.75,0.250000,-0.379094


In [16]:
# Advanced NLP preprocessing for ingredient text
import html
import unicodedata

# Lightweight fallback stopword list (can be replaced by NLTK stopwords if available)
DEFAULT_STOPWORDS = {
    'a','an','the','and','or','to','of','in','on','for','with','from','by',
    'at','as','is','are','was','were','be','been','being','it','this','that',
    'these','those','into','over','under','up','down','per','about','your','you'
}

def _normalize_unicode_nfkc(text: str) -> str:
    return unicodedata.normalize('NFKC', text)

def _normalize_punctuation(text: str) -> str:
    table = str.maketrans({
        '“': '"', '”': '"', '‘': "'", '’': "'",
        '–': '-', '—': '-', '−': '-',
    })
    return text.translate(table)

def _ascii_transliterate(text: str) -> str:
    # Keep simple ASCII for robust token matching
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('ascii')

def _html_and_web_cleanup(text: str) -> str:
    if not isinstance(text, str):
        return ''

    text = _normalize_unicode_nfkc(text)
    text = html.unescape(text)

    # Remove script/style blocks if any leaked content exists
    text = re.sub(r'<script[^>]*>.*?</script>', ' ', text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r'<style[^>]*>.*?</style>', ' ', text, flags=re.IGNORECASE | re.DOTALL)

    # Preserve basic structure separators before stripping tags
    text = re.sub(r'</?(h\d|p|div|li|ul|ol|br)[^>]*>', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'<a\s+[^>]*href=["\']([^"\']+)["\'][^>]*>(.*?)</a>', r'\2', text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove common boilerplate fragments
    boilerplate_patterns = [
        r'\bcookie(s)?\b', r'\bprivacy policy\b', r'\brelated posts\b',
        r'\bread more\b', r'\bshare\b', r'\bcopyright\b', r'\bsubscribe\b'
    ]
    for bp in boilerplate_patterns:
        text = re.sub(bp, ' ', text, flags=re.IGNORECASE)

    # Replace noisy web artifacts
    text = re.sub(r'https?://\S+|www\.\S+', ' <URL> ', text, flags=re.IGNORECASE)
    text = re.sub(r'\b[\w.+-]+@[\w-]+\.[\w.-]+\b', ' <EMAIL> ', text)
    text = re.sub(r'(?<!\w)@\w+', ' <HANDLE> ', text)
    text = re.sub(r'(?<!\w)#\w+', ' <HASHTAG> ', text)

    text = _normalize_punctuation(text)
    text = _ascii_transliterate(text)

    # Remove replacement chars / mixed encoding leftovers
    text = text.replace('\ufffd', ' ')

    # De-duplicate repeated lines
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    deduped = []
    seen = set()
    for ln in lines:
        if ln not in seen:
            deduped.append(ln)
            seen.add(ln)
    text = ' \n '.join(deduped)

    # Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\s*\n\s*', ' \n ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def _tokenize_ingredients(text: str):
    # Keep fractions and decimals: 3/4, 1.5, etc.; keep hyphenated words like fat-skimmed
    pattern = r'\d+/\d+|\d+\.\d+|\d+|[A-Za-z]+(?:[-\'][A-Za-z]+)?'
    return re.findall(pattern, text)

def _normalize_token(token: str, lowercase: bool = True):
    t = token.lower() if lowercase else token
    return t

def _remove_punctuation_with_care(tokens):
    # Tokenizer already drops most punctuation while preserving useful numeric fractions
    return [t for t in tokens if t.strip()]

def _remove_stopwords(tokens, stopwords_set=DEFAULT_STOPWORDS):
    return [t for t in tokens if t not in stopwords_set]

def _try_nltk_processors():
    try:
        import nltk
        from nltk import pos_tag
        from nltk.stem import WordNetLemmatizer

        # Try to ensure resources; if unavailable and cannot download, fallback gracefully
        needed = [
            ('tokenizers/punkt', 'punkt'),
            ('taggers/averaged_perceptron_tagger', 'averaged_perceptron_tagger'),
            ('corpora/wordnet', 'wordnet'),
            ('corpora/omw-1.4', 'omw-1.4'),
        ]
        for path_name, pkg in needed:
            try:
                nltk.data.find(path_name)
            except LookupError:
                try:
                    nltk.download(pkg, quiet=True)
                except Exception:
                    pass

        lemmatizer = WordNetLemmatizer()

        def _to_wordnet_pos(treebank_tag):
            if treebank_tag.startswith('J'):
                return 'a'
            if treebank_tag.startswith('V'):
                return 'v'
            if treebank_tag.startswith('N'):
                return 'n'
            if treebank_tag.startswith('R'):
                return 'r'
            return 'n'

        def pos_and_lemma(tokens):
            try:
                tags = pos_tag(tokens)
            except Exception:
                return [], tokens

            lemmas = [
                lemmatizer.lemmatize(tok, _to_wordnet_pos(tag))
                for tok, tag in tags
            ]
            return tags, lemmas

        return pos_and_lemma
    except Exception:
        return None

nltk_processor = _try_nltk_processors()

def preprocess_text_for_nlp(text: str, lowercase: bool = True, remove_stopwords: bool = True):
    cleaned = _html_and_web_cleanup(text)

    tokens = _tokenize_ingredients(cleaned)
    tokens = [_normalize_token(t, lowercase=lowercase) for t in tokens]
    tokens = _remove_punctuation_with_care(tokens)

    if remove_stopwords:
        tokens_no_stop = _remove_stopwords(tokens)
    else:
        tokens_no_stop = tokens

    pos_tags = []
    lemmas = tokens_no_stop
    if nltk_processor is not None and len(tokens_no_stop) > 0:
        pos_tags, lemmas = nltk_processor(tokens_no_stop)

    return {
        'ingredients_clean': cleaned,
        'tokens': tokens,
        'tokens_no_stop': tokens_no_stop,
        'lemmas': lemmas,
        'pos_tags': pos_tags,
    }

# Apply NLP preprocessing to ingredient field
processed = df['ingredients'].fillna('').apply(
    lambda x: preprocess_text_for_nlp(x, lowercase=True, remove_stopwords=True)
)

df['ingredients_clean'] = processed.map(lambda d: d['ingredients_clean'])
df['tokens'] = processed.map(lambda d: d['tokens'])
df['tokens_no_stop'] = processed.map(lambda d: d['tokens_no_stop'])
df['lemmas'] = processed.map(lambda d: d['lemmas'])
df['pos_tags'] = processed.map(lambda d: d['pos_tags'])
df['token_count'] = df['tokens'].map(len)

# Print processed examples
example_cols = [
    'ingredients', 'ingredients_clean', 'tokens_no_stop', 'lemmas', 'pos_tags',
    'servings', 'sugar_total_g', 'sugar_per_serving_g', 'sugar_per_serving_z'
]
print('NLP processed examples:')
for i, row in df[example_cols].head(3).iterrows():
    print('\n' + '=' * 60)
    print(f'Example index: {i}')
    print('ORIGINAL:', row['ingredients'])
    print('CLEAN   :', row['ingredients_clean'])
    print('TOKENS  :', row['tokens_no_stop'][:25])
    print('LEMMAS  :', row['lemmas'][:25])
    if row['pos_tags']:
        print('POS TAGS:', row['pos_tags'][:15])
    else:
        print('POS TAGS: skipped (NLTK resources unavailable)')

NLP processed examples:

Example index: 0
ORIGINAL: 2 cups Baby Bella mushrooms, sliced, 2 cups arborio rice, 1 tsp olive oil, 3 tsp butter, 2 shallots, minced, 1 cup white wine, 8 cups fat free chicken stock (or vegetable stock), salt and pepper, salt and pepper, 1/2 cup grated parmesan cheese, 4 tbsp chopped parsley
CLEAN   : 2 cups Baby Bella mushrooms, sliced, 2 cups arborio rice, 1 tsp olive oil, 3 tsp butter, 2 shallots, minced, 1 cup white wine, 8 cups fat free chicken stock (or vegetable stock), salt and pepper, salt and pepper, 1/2 cup grated parmesan cheese, 4 tbsp chopped parsley
TOKENS  : ['2', 'cups', 'baby', 'bella', 'mushrooms', 'sliced', '2', 'cups', 'arborio', 'rice', '1', 'tsp', 'olive', 'oil', '3', 'tsp', 'butter', '2', 'shallots', 'minced', '1', 'cup', 'white', 'wine', '8']
LEMMAS  : ['2', 'cups', 'baby', 'bella', 'mushrooms', 'sliced', '2', 'cups', 'arborio', 'rice', '1', 'tsp', 'olive', 'oil', '3', 'tsp', 'butter', '2', 'shallots', 'minced', '1', 'cup', 'white', '

In [17]:
# Optional modern NLP features: Hugging Face tokenizer + TF-IDF features
from tokenizers import Tokenizer
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer

sample_texts = df['ingredients_clean'].fillna('').tolist()

# 1) Hugging Face tokenizer (subword tokenization)
hf_tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
hf_batch = hf_tokenizer(
    sample_texts[:3],
    truncation=True,
    padding='max_length',
    max_length=64,
    return_tensors='np'
 )
print('HF tokenizer example shapes:', {k: v.shape for k, v in hf_batch.items()})

# 2) Classic NLP sparse features (TF-IDF)
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=5000,
    token_pattern=r'(?u)\b\w+\b',
 )
X_tfidf = tfidf.fit_transform(sample_texts)
print('TF-IDF matrix shape:', X_tfidf.shape)
print('Sample feature names:', tfidf.get_feature_names_out()[:20])

'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/config.json
Retrying in 1s [Retry 1/5].


OSError: Can't load the configuration of 'distilbert-base-uncased'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'distilbert-base-uncased' is the correct path to a directory containing a config.json file

In [ ]:
df.to_csv('sugar.csv', index=False)
print('Saved sugar.csv with columns:', list(df.columns))

In [ ]:
from huggingface_hub import notebook_login

# Login interactively (do not hardcode tokens in notebooks)
notebook_login()

In [ ]:
dataset = load_dataset("csv", data_files="sugar.csv")

In [ ]:
dataset = dataset['train'].shuffle(seed=42).train_test_split(test_size=0.08)

In [ ]:
dataset

In [ ]:
# Example
# dataset.push_to_hub("ziq/ingredient_to_sugar_level")
dataset.push_to_hub("path/name_for_your_dataset")